# Micro Proyecto 1: Modelos avanzados para el Procesamiento de Lenguaje Natural, Grupo 

---

## Integrantes:

1.
2.
3.
4. Mateo Hernández Gualdrón, 202422619


**Objetivo:** Entrenar un modelo de redes neuronales con la producción musical de un único compositor para generar secuencias musicales en su estilo. El dataset original cuenta con 19 reconocidos compositores.

**Características Extraídas:**
* `pitch` (categórica): Representa la nota musical.
* `step` (numérica): Tiempo entre el inicio de la nota anterior y la actual.
* `duration` (numérica): Diferencia entre el fin y el inicio de la nota.
  
---





In [3]:
import os
import glob
import numpy as np
import pretty_midi
import torch
import torch.nn as torch_nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

# Configuración del dispositivo (GPU si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cpu


# Preprocesamiento

In [ ]:
def extract_midi_features(midi_path, instrument_idx=0):
    """
    Extrae pitch, step y duration de un archivo MIDI usando pretty_midi.
    """
    try:
        pm = pretty_midi.PrettyMIDI(midi_path)
        if len(pm.instruments) <= instrument_idx:
            return None
        
        instrument = pm.instruments[instrument_idx]
        notes = sorted(instrument.notes, key=lambda note: note.start)
        
        features = []
        prev_start = 0.0
        
        for note in notes:
            pitch = note.pitch # Categórica
            step = note.start - prev_start # Numérica
            duration = note.end - note.start # Numérica
            
            features.append([pitch, step, duration])
            prev_start = note.start
            
        return features
    except Exception as e:
        print(f"Error procesando {midi_path}: {e}")
        return None

def load_composer_data(base_dir, composer_name):
    """
    Lee todos los MIDIs de un compositor con rutas dinámicas usando os.
    """
    composer_path = os.path.join(base_dir, composer_name)
    # Busca tanto .mid como .midi
    midi_files = glob.glob(os.path.join(composer_path, '*.mid*'))
    
    all_sequences = []
    for f in midi_files:
        feats = extract_midi_features(f)
        if feats and len(feats) > 10: # Filtrar secuencias muy cortas
            all_sequences.append(feats)
            
    return all_sequences

DATASET_DIR = os.path.join("dataset", "music_artist")
COMPOSER = "beeth" 
sequences = load_composer_data(DATASET_DIR, COMPOSER)
print(f"Total de secuencias válidas procesadas para {COMPOSER}: {len(sequences)}")

Total de secuencias válidas procesadas para beeth: 29


# Loader

In [8]:
class MusicDataset(Dataset):
    def __init__(self, sequences, seq_length=25):
        self.seq_length = seq_length
        self.inputs = []
        self.targets = []
        self.vocab_size = 128 # Rango estándar MIDI para pitch
        
        self._build_dataset(sequences)
        
    def _build_dataset(self, sequences):
        for seq in sequences:
            for i in range(len(seq) - self.seq_length):
                # Contexto: c notas anteriores [cite: 59]
                self.inputs.append(seq[i:i + self.seq_length])
                # Predicción: la siguiente nota en el tiempo t [cite: 59]
                self.targets.append(seq[i + self.seq_length])
                
        self.inputs = torch.tensor(self.inputs, dtype=torch.float32)
        self.targets = torch.tensor(self.targets, dtype=torch.float32)
        
    def __len__(self):
        return len(self.inputs)
    
    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

SEQ_LENGTH = 25
BATCH_SIZE = 64

music_dataset = MusicDataset(sequences, seq_length=SEQ_LENGTH)
dataloader = DataLoader(music_dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f"Tamaño del dataset de entrenamiento: {len(music_dataset)} ventanas de contexto.")

Tamaño del dataset de entrenamiento: 52667 ventanas de contexto.


## Arquitectura del Modelo
[cite_start]
Se implementa una arquitectura recurrente desde cero haciendo uso de PyTorch (Embeddings y LSTM)[cite: 117]. 
Al tener que predecir variables de distinta naturaleza, la red se divide en múltiples cabezales de salida. [cite_start]
La función de pérdida compuesta es la suma de una entropía cruzada (clasificación) y errores cuadráticos medios (regresión)[cite: 22].

$$Loss_{total} = CrossEntropy(Pitch) + MSE(Step) + MSE(Duration)$$

In [10]:
class MusicGeneratorLSTM(torch_nn.Module):
    def __init__(self, vocab_size=128, embed_dim=64, hidden_dim=128):
        super(MusicGeneratorLSTM, self).__init__()
        
        # Procesamiento de entradas
        self.pitch_embedding = torch_nn.Embedding(vocab_size, embed_dim)
        self.step_proj = torch_nn.Linear(1, 16)
        self.dur_proj = torch_nn.Linear(1, 16)
        
        # Capa recurrente (tamaño entrada = embed_dim + 16 + 16)
        lstm_input_dim = embed_dim + 32
        self.lstm = torch_nn.LSTM(lstm_input_dim, hidden_dim, num_layers=2, batch_first=True)
        
        # Cabezales de salida múltiple
        self.pitch_out = torch_nn.Linear(hidden_dim, vocab_size)
        self.step_out = torch_nn.Linear(hidden_dim, 1)
        self.duration_out = torch_nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        # x shape: (batch, seq_length, 3) -> [pitch, step, duration]
        pitch = x[:, :, 0].long()
        step = x[:, :, 1].unsqueeze(-1)
        duration = x[:, :, 2].unsqueeze(-1)
        
        # Proyectar características
        p_emb = self.pitch_embedding(pitch)
        s_emb = torch.relu(self.step_proj(step))
        d_emb = torch.relu(self.dur_proj(duration))
        
        # Concatenar para la LSTM
        lstm_in = torch.cat([p_emb, s_emb, d_emb], dim=-1)
        
        # Pasar por LSTM
        lstm_out, _ = self.lstm(lstm_in)
        
        # Tomar la última salida temporal para la predicción
        last_hidden = lstm_out[:, -1, :]
        
        # Predicciones
        pitch_pred = self.pitch_out(last_hidden)
        step_pred = torch.relu(self.step_out(last_hidden)) # ReLU para evitar tiempos negativos
        duration_pred = torch.relu(self.duration_out(last_hidden))
        
        return pitch_pred, step_pred, duration_pred

model = MusicGeneratorLSTM().to(device)

In [ ]:
# Hiperparámetros y Optimizador
epochs = 100  # Aumentamos el límite de épocas
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Función de costo compuesta # [cite: 22, 105]
criterion_pitch = torch_nn.CrossEntropyLoss()
criterion_mse = torch_nn.MSELoss()

# Pesos para balancear la función de pérdida (Loss Weighting)
lambda_pitch = 1.0
lambda_step = 10.0
lambda_dur = 10.0

# Configuración de Early Stopping y Checkpointing
patience = 5
best_loss = float('inf')
epochs_no_improve = 0
best_model_path = 'mejor_modelo_compositor.pth'

model.train()
for epoch in range(epochs):
    epoch_loss = 0
    for batch_idx, (inputs, targets) in enumerate(dataloader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        pitch_pred, step_pred, dur_pred = model(inputs)
        
        # Extraer targets
        target_pitch = targets[:, 0].long()
        target_step = targets[:, 1].unsqueeze(-1)
        target_dur = targets[:, 2].unsqueeze(-1)
        
        # Calcular pérdidas individuales
        loss_pitch = criterion_pitch(pitch_pred, target_pitch)
        loss_step = criterion_mse(step_pred, target_step)
        loss_dur = criterion_mse(dur_pred, target_dur)
        
        # Pérdida compuesta ponderada # [cite: 22, 105]
        loss = (lambda_pitch * loss_pitch) + (lambda_step * loss_step) + (lambda_dur * loss_dur)
        
        # Backward y optimización
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{epochs} | Loss Promedio: {avg_loss:.4f}")
    
    # Lógica de Early Stopping
    if avg_loss < best_loss:
        best_loss = avg_loss
        epochs_no_improve = 0
        # Guardar los pesos del mejor modelo
        torch.save(model.state_dict(), best_model_path)
    else:
        epochs_no_improve += 1
        print(f"   -> Sin mejora. Paciencia: {epochs_no_improve}/{patience}")
        if epochs_no_improve >= patience:
            print(f"\n¡Early stopping activado en la época {epoch+1}! Recuperando el mejor modelo...")
            break

model.load_state_dict(torch.load(best_model_path))
print(f"Entrenamiento finalizado. Mejor Loss validado: {best_loss:.4f}")

Epoch 1/10 | Loss: 4.5960
Epoch 2/10 | Loss: 4.1488
Epoch 3/10 | Loss: 3.9290
Epoch 4/10 | Loss: 3.7149
Epoch 5/10 | Loss: 3.5118
Epoch 6/10 | Loss: 3.2954
Epoch 7/10 | Loss: 3.0949
Epoch 8/10 | Loss: 2.9067
Epoch 9/10 | Loss: 2.7150
Epoch 10/10 | Loss: 2.5459


In [ ]:
def generate_music(model, seed_sequence, num_notes=200):
    """
    Genera una secuencia de música iterando sobre sus propias predicciones.
    Debe generar 200 notas para los entregables.
    """
    model.eval()
    generated_seq = list(seed_sequence)
    current_input = torch.tensor([seed_sequence], dtype=torch.float32).to(device)
    
    with torch.no_grad():
        for _ in range(num_notes):
            # Predecir siguiente nota
            p_pred, s_pred, d_pred = model(current_input)
            
            # Tomar el pitch con mayor probabilidad (Greedy Decoding)
            next_pitch = torch.argmax(p_pred, dim=-1).item()
            next_step = s_pred.item()
            next_dur = d_pred.item()
            
            next_note = [next_pitch, next_step, next_dur]
            generated_seq.append(next_note)
            
            # Actualizar ventana deslizante
            current_input = torch.tensor([generated_seq[-SEQ_LENGTH:]], dtype=torch.float32).to(device)
            
    return generated_seq[len(seed_sequence):] # Retornar solo lo generado

def create_midi_file(generated_notes, output_path="output.mid"):
    """
    Reconstruye el archivo MIDI a partir de las predicciones.
    """
    pm = pretty_midi.PrettyMIDI() 
    instrument = pretty_midi.Instrument(program=0) # Piano 
    
    current_time = 0.0
    for feat in generated_notes:
        pitch, step, duration = feat
        
        start = current_time + step
        end = start + duration
        current_time = start
        
        note = pretty_midi.Note(velocity=100, pitch=int(pitch), start=start, end=end) 
        instrument.notes.append(note) 
        
    pm.instruments.append(instrument) 
    pm.write(output_path) 
    print(f"Archivo guardado en {output_path}")

# Ejecución de la inferencia para 3 composiciones distintas
if len(sequences) >= 3:
    os.makedirs('outputs', exist_ok=True)
    for i in range(3):
        print(f"Generando composición {i+1}...")
        seed = sequences[i][:SEQ_LENGTH] 
        new_notes = generate_music(model, seed, num_notes=200) 
        
        # Generar y guardar el archivo MIDI
        create_midi_file(new_notes, output_path=f"outputs/composicion_generada_{i+1}.mid")
else:
    print("No hay suficientes secuencias válidas en el dataset para tomar 3 semillas distintas.")

Generando composición 1...
Archivo guardado en outputs/composicion_generada_1.mid
Generando composición 2...
Archivo guardado en outputs/composicion_generada_2.mid
Generando composición 3...
Archivo guardado en outputs/composicion_generada_3.mid
